# Trabalho 1 — Aquisição de Dados

## 1. Identificação do projeto

**Tema:**

Análise da relação entre a variação dos preços da cesta básica, a inflação dos alimentos e os diferentes períodos de governos presidenciais e estaduais no Brasil desde 1994.

**Integrantes:**

- Nome 1
- Nome 2
- Nome 3
- Nome 4

**Disciplina:** Ciência de Dados

**Instituição:** Universidade Federal do Amazonas — Instituto de Computação

## 2. Pergunta motivadora

De que maneira a variação nos preços da cesta básica e a inflação dos alimentos se
comportaram ao longo das diferentes gestões presidenciais e estaduais no Brasil
desde 1994, e como os ciclos eleitorais e espectros partidários se correlacionam com
esses momentos de instabilidade?

## 3. Objetivo da coleta

Construir uma base de dados integrada contendo informações sobre
a inflação dos alimentos, os preços da cesta básica e informações
relacionadas aos períodos de governos e eleições no Brasil.

A base será utilizada nas etapas posteriores do projeto para
investigar possíveis padrões entre as variáveis econômicas e
políticas.

## 4. Bibliotecas e configurações

### 4.1 Bibliotecas utilizadas

| Biblioteca | Finalidade | Instalação |
|---|---|---|
| `requests` | Requisições HTTP para a API do IBGE/SIDRA e tratamento de status HTTP | `pip install requests` |
| `pandas` | Leitura, limpeza, transformação e integração de dados; leitura de HTML com `read_html` | `pip install pandas` |
| `beautifulsoup4` | Parsing e extração de dados do HTML da página do DIEESE | `pip install beautifulsoup4` |
| `lxml` | Parser HTML/XML de alta performance (motor alternativo para BeautifulSoup) | `pip install lxml` |
| `pyarrow` | Leitura e escrita de arquivos Parquet (formato recomendado para a base tratada) | `pip install pyarrow` |
| `json` (padrão) | Processamento de respostas JSON da API | - |
| `datetime` (padrão) | Registro de data e hora das coletas com fuso horário | - |
| `time` (padrão) | Pausas entre requisições para respeitar limites dos servidores | - |
| `pathlib` (padrão) | Construção de caminhos de arquivos de forma portável | - |

### 4.2 Configurações iniciais

- **User-Agent:** identificar o robô em requisições HTTP, conforme recomendado em `robots.txt`.
- **Timeout:** definir limite de espera por requisição (ex: 30 segundos).
- **Pausas entre requisições:** intervalo de pelo menos 1 segundo entre chamadas à API e entre páginas scrapeadas.
- **Codificação de resposta:** forçar `utf-8` ao ler HTML ou JSON para evitar problemas de acentuação.
- **Seed de data/hora:** registrar cada coleta com `datetime.now(timezone.utc)` ou fuso horário de Brasilia (`America/Sao_Paulo`).


In [ ]:
# Bibliotecas e configurações
import requests
import pandas as pd
import json
import os
from datetime import datetime, timezone
from pathlib import Path
import time

# Parser HTML
from bs4 import BeautifulSoup

# Exportacao Parquet
import pyarrow as pa
import pyarrow.parquet as pq

# Configuracoes gerais
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# Fuso horario de referencia (Brasilia)
FUSO = "America/Sao_Paulo"

# Caminhos base (ajuste conforme a estrutura da sua maquina)
BASE_DIR = Path.cwd()
BRUTOS_DIR = BASE_DIR / "projeto" / "dados_brutos"
TRATADOS_DIR = BASE_DIR / "projeto" / "dados_tratados"
DOC_DIR = BASE_DIR / "projeto" / "documentacao"

# Criar pastas se nao existirem
BRUTOS_DIR.mkdir(parents=True, exist_ok=True)
TRATADOS_DIR.mkdir(parents=True, exist_ok=True)
DOC_DIR.mkdir(parents=True, exist_ok=True)

# Configuracoes de requisicao
HEADERS = {
    "User-Agent": "TrabalhoAcademico-CienciaDeDados/1.0 (contato@exemplo.com)"
}
TIMEOUT = 30  # segundos
PAUSA = 1.5   # segundos entre requisicoes

print("Bibliotecas carregadas e pastas configuradas.")


## 5. Fonte 1 — IBGE/SIDRA (API)

### 5.1 Descrição da fonte

O Sistema IBGE de Recuperação Automática (SIDRA) é uma plataforma do Instituto Brasileiro de Geografia e Estatística (IBGE) que disponibiliza dados estatísticos oficiais do Brasil. Para este trabalho, será utilizada a API do SIDRA para adquirir dados relacionados ao Índice Nacional de Preços ao Consumidor Amplo (IPCA), com foco nos preços e na inflação dos alimentos. Esses dados serão utilizados para analisar a evolução dos preços ao longo do período estudado e posteriormente integrados aos dados obtidos por Web Scraping.


### 5.2 Definição dos dados

Será utilizada a **Tabela 61 do Sistema Nacional de Índices de Preços ao Consumidor (SNIPC)**, disponibilizada pelo IBGE por meio do SIDRA:

- **Indicador:** Índice Nacional de Preços ao Consumidor Amplo (IPCA);
- **Tabela:** 61 — *IPCA — Peso mensal, para o índice geral, grupos, subgrupos, itens e subitens de produtos e serviços*;
- **Período disponível na tabela:** janeiro de 1991 a julho de 1999;
- **Dimensão temporal:** mês e ano de referência;
- **Dimensão dos produtos e serviços:** índice geral, grupos, subgrupos, itens e subitens;
- **Variável principal:** peso mensal de cada grupo, subgrupo, item ou subitem na composição do IPCA;
- **Unidade de observação:** uma categoria de produto ou serviço observada em determinado mês e ano;
- **Finalidade na pesquisa:** representar a participação relativa dos alimentos e de outros grupos de consumo na composição do IPCA, permitindo comparar a evolução da inflação dos alimentos com os períodos de governos e ciclos eleitorais.

A tabela será utilizada como fonte de informações sobre a estrutura mensal do IPCA no período inicial da série histórica analisada. Para a comparação com os preços da cesta básica, será necessário selecionar as categorias relacionadas à alimentação e documentar os códigos, níveis e classificações escolhidos no SIDRA.

**Observação sobre a cobertura temporal:** a Tabela 61 termina em julho de 1999. Como o projeto pretende analisar o período desde 1994 e pode exigir dados posteriores a 1999, será necessário verificar, em etapa posterior, se outras tabelas do SIDRA devem ser incorporadas para complementar a série do IPCA e manter a cobertura temporal do estudo.

**Fonte da definição:** [Tabela 61 — IPCA no SIDRA](https://sidra.ibge.gov.br/Tabela/61), incluindo as [notas da tabela](https://sidra.ibge.gov.br/Tabela/61#notas-tabela).

### 5.3 URL da API

[https://sidra.ibge.gov.br/Tabela/61](https://sidra.ibge.gov.br/Tabela/61)



In [ ]:
### 5.3 Teste da requisição
import requests

url = ""

resposta = requests.get(url)

print(resposta.status_code)
print(resposta.text)

In [ ]:
# 5.5 Coleta completa
# Preencher posteriormente.

In [ ]:
# 5.6 Salvamento dos dados brutos
# Preencher posteriormente.

## 6. Fonte 2 — DIEESE (Web Scraping)

### 6.1 Descrição da fonte

*Preencher posteriormente.*

### 6.2 URL

*Preencher posteriormente.*

### 6.3 Estrutura HTML

*Preencher posteriormente.*

In [ ]:
# 6.4 Scraping
# Preencher posteriormente.

In [ ]:
# 6.5 Salvamento dos dados brutos
# Preencher posteriormente.

## 7. Tratamento dos dados

### 7.1 Tratamento IBGE

*Preencher posteriormente.*

### 7.2 Tratamento DIEESE

*Preencher posteriormente.*

### 7.3 Padronização

*Preencher posteriormente.*

In [ ]:
# Tratamento e padronização dos dados
# Preencher posteriormente.

## 8. Integração das fontes

### 8.1 Chave de integração

*Preencher posteriormente.*

### 8.2 Junção

*Preencher posteriormente.*

### 8.3 Verificação

*Preencher posteriormente.*

In [ ]:
# Integração e verificação
# Preencher posteriormente.

## 9. Base final

### 9.1 Estrutura

*Preencher posteriormente.*

### 9.2 Quantidade de registros

*Preencher posteriormente.*

### 9.3 Exportação

*Preencher posteriormente.*

In [ ]:
# Exportação da base final
# Preencher posteriormente.

## 10. Proveniência e observações

*Preencher posteriormente.*